<a href="https://colab.research.google.com/github/svelasquezso-pixel/trabajos_poo/blob/main/trabajo62.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
from datetime import datetime, timedelta


habitaciones = []
FILE_NAME = "hotel_data.txt"


PRECIOS = {
    '1': 120000, '2': 120000, '3': 120000, '4': 120000, '5': 120000,
    '6': 160000, '7': 160000, '8': 160000, '9': 160000, '10': 160000
}

def inicializar_habitaciones():
    """Inicializa la lista de habitaciones al inicio de la aplicación."""
    for i in range(1, 11):
        num = str(i)
        habitaciones.append({
            'numero': num,
            'precio': PRECIOS[num],
            'estado': 'Disponible',
            'huesped': None  # {nombre, apellido, dni, fecha_ingreso (datetime)}
        })

def cargar_datos_hotel():
    """Carga el estado de las habitaciones desde el archivo, si existe."""
    if not os.path.exists(FILE_NAME):
        open(FILE_NAME, "w", encoding="utf-8").close()
        inicializar_habitaciones()
        return

    try:
        with open(FILE_NAME, "r", encoding="utf-8") as file:
            lineas = file.readlines()

        if not lineas:
            inicializar_habitaciones()
            return

        global habitaciones
        habitaciones = []

        for linea in lineas:
            datos = linea.strip().split("|")
            if len(datos) == 7:
                num, precio, estado, nombre, apellido, dni, fecha_ingreso_str = datos
                hab = {
                    'numero': num,
                    'precio': int(precio),
                    'estado': estado,
                }
                if estado == 'Ocupada':
                    hab['huesped'] = {
                        'nombre': nombre,
                        'apellido': apellido,
                        'dni': dni,
                        'fecha_ingreso': datetime.strptime(fecha_ingreso_str, "%Y-%m-%d")
                    }
                else:
                    hab['huesped'] = None
                habitaciones.append(hab)
            # Manejar el caso de una línea vacía o mal formateada
            elif len(datos) == 3:
                 num, precio, estado = datos
                 habitaciones.append({
                    'numero': num,
                    'precio': int(precio),
                    'estado': estado,
                    'huesped': None
                })

        # Si el archivo está vacío o mal formado, inicializar de nuevo
        if not habitaciones:
            inicializar_habitaciones()

    except Exception as e:
        messagebox.showerror("Error de Carga", f"Fallo al cargar datos: {e}. Se inicializarán las habitaciones.")
        inicializar_habitaciones()


cargar_datos_hotel()



def guardar_datos_hotel():
    """Guarda el estado actual de las habitaciones en el archivo."""
    try:
        with open(FILE_NAME, "w", encoding="utf-8") as file:
            for hab in habitaciones:
                linea = f"{hab['numero']}|{hab['precio']}|{hab['estado']}"
                if hab['estado'] == 'Ocupada' and hab['huesped']:
                    huesped = hab['huesped']

                    fecha_str = huesped['fecha_ingreso'].strftime("%Y-%m-%d")
                    linea += f"|{huesped['nombre']}|{huesped['apellido']}|{huesped['dni']}|{fecha_str}"
                file.write(linea + "\n")
        messagebox.showinfo("Completado", "Estado del hotel guardado exitosamente.")
    except Exception as e:
        messagebox.showerror("Error", f"Fallo al guardar los datos: {e}")


def registrar_ingreso(habitacion_num, fecha_ingreso_str, nombre, apellido, dni, win_registro):
    """
    Función que valida los datos del huésped, registra el ingreso y actualiza el estado.
    """


    if not all([fecha_ingreso_str.strip(), nombre.strip(), apellido.strip(), dni.strip()]):
        messagebox.showerror("Fallo", "Todos los campos son obligatorios.")
        return


    try:
        fecha_ingreso = datetime.strptime(fecha_ingreso_str.strip(), "%Y-%m-%d")
    except ValueError:
        messagebox.showerror("Error de Validación", "Formato de fecha de ingreso incorrecto. Use YYYY-MM-DD.")
        return


    for hab in habitaciones:
        if hab['numero'] == habitacion_num:
            hab['estado'] = 'Ocupada'
            hab['huesped'] = {
                'nombre': nombre.strip(),
                'apellido': apellido.strip(),
                'dni': dni.strip(),
                'fecha_ingreso': fecha_ingreso
            }
            messagebox.showinfo("Registro Exitoso",
                                f"Huésped {nombre} {apellido} registrado en Habitación {habitacion_num}. Precio: {hab['precio']:,} por día.")
            win_registro.destroy()
            return

--------------------------------------------------------

def abrir_registro_huesped(habitacion_num, win_listado):
    """Genera la ventana para ingresar los datos del huésped."""


    win_listado.destroy()

    win = tk.Toplevel(root)
    win.title(f"Registro de Ingreso - Habitación {habitacion_num}")
    win.transient(root)
    win.grab_set()

    input_frame = tk.Frame(win, padx=10, pady=10)
    input_frame.pack(expand=True, fill="both")


    precio = next(hab['precio'] for hab in habitaciones if hab['numero'] == habitacion_num)
    tk.Label(input_frame, text=f"Habitación {habitacion_num}: Precio/día: ${precio:,}").grid(row=0, column=0, columnspan=2, pady=5)


    def create_input(parent, text, row, default_text=""):
        tk.Label(parent, text=text + ":").grid(row=row, column=0, padx=5, pady=2, sticky="w")
        entry = tk.Entry(parent)
        if default_text:
             entry.insert(0, default_text)
        entry.grid(row=row, column=1, padx=5, pady=2, sticky="ew")
        return entry


    fecha_ingreso = create_input(input_frame, "Fecha Ingreso (YYYY-MM-DD)", 1, datetime.now().strftime("%Y-%m-%d"))
    nombre = create_input(input_frame, "Nombre", 2, "Nombre")
    apellido = create_input(input_frame, "Apellido", 3, "Apellido")
    dni = create_input(input_frame, "Documento de Identidad", 4, "DNI")


    boton1 = tk.Button(input_frame, text="Registrar Ingreso",
                       command=lambda: registrar_ingreso(
                            habitacion_num,
                            fecha_ingreso.get(),
                            nombre.get(),
                            apellido.get(),
                            dni.get(),
                            win))
    boton1.grid(row=5, column=0, columnspan=2, pady=10)


def consultar_habitaciones():
    """Muestra el listado de habitaciones y permite seleccionar una para ocupar."""
    win = tk.Toplevel(root)
    win.title("Consultar Habitaciones")
    win.transient(root)
    win.grab_set()

    tk.Label(win, text="Seleccione una Habitación Disponible para Ocupar").pack(pady=5)

    listado_frame = tk.Frame(win, padx=5, pady=5)
    listado_frame.pack(pady=5)


    tk.Label(listado_frame, text="Habitación", relief="ridge").grid(row=0, column=0, padx=5)
    tk.Label(listado_frame, text="Estado", relief="ridge").grid(row=0, column=1, padx=5)
    tk.Label(listado_frame, text="Acción", relief="ridge").grid(row=0, column=2, padx=5)

    for i, hab in enumerate(habitaciones):
        row_index = i + 1


        tk.Label(listado_frame,
                 text=f"Habitación {hab['numero']} (${hab['precio']:,}/día)").grid(row=row_index, column=0, padx=5, pady=2, sticky="w")


        tk.Label(listado_frame,
                 text=hab['estado']).grid(row=row_index, column=1, padx=5, pady=2)


        if hab['estado'] == 'Disponible':
            btn = tk.Button(listado_frame, text="Ocupar",
                            command=lambda num=hab['numero']: abrir_registro_huesped(num, win))
            btn.grid(row=row_index, column=2, padx=5, pady=2)
        else:

            nombre = hab['huesped']['nombre'] if hab['huesped'] else "N/A"
            tk.Label(listado_frame, text=f"Ocupada por: {nombre}").grid(row=row_index, column=2, padx=5, pady=2)

------------------------------------------------------

def calcular_salida(hab_data, fecha_salida_str, win_calculo, total_label, dias_label, boton_registro):
    """Calcula el total a pagar y habilita el botón de registro de salida."""
    try:
        fecha_ingreso = hab_data['huesped']['fecha_ingreso']
        fecha_salida = datetime.strptime(fecha_salida_str.strip(), "%Y-%m-%d")
    except ValueError:
        messagebox.showerror("Error de Fecha", "Formato de fecha de salida incorrecto. Use YYYY-MM-DD.")
        return


    if fecha_salida <= fecha_ingreso:
        messagebox.showerror("Error de Validación", "La fecha de salida debe ser posterior a la fecha de ingreso.")

        boton_registro.config(state=tk.DISABLED)
        return


    dias = (fecha_salida - fecha_ingreso).days

    dias_totales = max(1, dias)

    precio_total = dias_totales * hab_data['precio']

    dias_label.config(text=f"Total de días de alojamiento: {dias_totales} días")
    total_label.config(text=f"TOTAL A PAGAR: ${precio_total:,.2f}")


    boton_registro.config(state=tk.NORMAL,
                          command=lambda: registrar_salida_final(hab_data['numero'], win_calculo))

def registrar_salida_final(habitacion_num, win_calculo):
    """Registra la salida, limpia el huésped y pone la habitación como 'Disponible'."""
    for hab in habitaciones:
        if hab['numero'] == habitacion_num:
            hab['estado'] = 'Disponible'
            nombre_huesped = hab['huesped']['nombre']
            hab['huesped'] = None
            messagebox.showinfo("Salida Registrada",
                                f"Salida de {nombre_huesped} registrada. Habitación {habitacion_num} ahora está Disponible.")
            win_calculo.destroy()
            return

def abrir_registro_salida(habitacion_num, win_inicial):
    """Genera la ventana para ingresar la fecha de salida y calcular el pago."""


    win_inicial.destroy()

    hab_data = next(hab for hab in habitaciones if hab['numero'] == habitacion_num)
    huesped = hab_data['huesped']

    win = tk.Toplevel(root)
    win.title(f"Salida de Huésped - Habitación {habitacion_num}")
    win.transient(root)
    win.grab_set()

    main_frame = tk.Frame(win, padx=10, pady=10)
    main_frame.pack(expand=True, fill="both")


    tk.Label(main_frame, text=f"Habitación a Entregar: {habitacion_num}").pack(pady=5)
    tk.Label(main_frame, text=f"Huésped: {huesped['nombre']} {huesped['apellido']} (DNI: {huesped['dni']})").pack()
    tk.Label(main_frame, text=f"Fecha de Ingreso: {huesped['fecha_ingreso'].strftime('%Y-%m-%d')}").pack(pady=5)


    tk.Label(main_frame, text="Fecha Salida (YYYY-MM-DD):").pack(pady=5)
    fecha_salida = tk.Entry(main_frame)

    fecha_salida.insert(0, datetime.now().strftime("%Y-%m-%d"))
    fecha_salida.pack()


    dias_label = tk.Label(main_frame, text="Total de días de alojamiento: N/A")
    dias_label.pack(pady=5)
    total_label = tk.Label(main_frame, text="TOTAL A PAGAR: $0.00")
    total_label.pack(pady=5)


    btn_calcular = tk.Button(main_frame, text="Calcular Total a Pagar",
                             command=lambda: calcular_salida(
                                hab_data,
                                fecha_salida.get(),
                                win,
                                total_label,
                                dias_label,
                                btn_registro))
    btn_calcular.pack(pady=5)


    btn_registro = tk.Button(main_frame, text="Registrar Salida",
                             state=tk.DISABLED)
    btn_registro.pack(pady=5)

def salida_huespedes():
    """Ventana inicial que solicita el número de habitación a entregar."""
    win = tk.Toplevel(root)
    win.title("Salida de Huésped")
    win.transient(root)
    win.grab_set()

    input_frame = tk.Frame(win, padx=10, pady=10)
    input_frame.pack(expand=True, fill="both")

    tk.Label(input_frame, text="Número de Habitación a Entregar (1-10):").grid(row=0, column=0, padx=5, pady=5)
    num_hab_entry = tk.Entry(input_frame)
    num_hab_entry.grid(row=0, column=1, padx=5, pady=5)

    def validar_habitacion_salida():
        num_str = num_hab_entry.get().strip()
        if not num_str.isdigit() or not (1 <= int(num_str) <= 10):
            messagebox.showerror("Error", "Número de habitación inválido (debe ser entre 1 y 10).")
            return

        hab = next((h for h in habitaciones if h['numero'] == num_str), None)

        if hab is None or hab['estado'] == 'Disponible':
            messagebox.showerror("Error", f"La habitación {num_str} no está ocupada.")
            return


        abrir_registro_salida(num_str, win)

    boton1 = tk.Button(input_frame, text="Continuar", command=validar_habitacion_salida)
    boton1.grid(row=1, column=0, columnspan=2, pady=10)



root = tk.Tk()
root.title("Gestión de Hotel - 10 Habitaciones")


header = tk.Frame(root, pady=5)
header.pack(side="top", fill="x")


boton_consultar = tk.Button(header, text="Consultar habitaciones", command=consultar_habitaciones)
boton_consultar.pack(side="left", padx=5)

boton_salida = tk.Button(header, text="Salida de huéspedes", command=salida_huespedes)
boton_salida.pack(side="left", padx=5)

boton_guardar = tk.Button(header, text="Guardar Estado", command=guardar_datos_hotel)
boton_guardar.pack(side="left", padx=5)

root.mainloop()